# Baseline PyTorch — Classification binaire (tumeur vs non-tumeur)

Objectif : entraîner un **modèle baseline** simple sur le dataset Kaggle MRI (tumeur / non-tumeur), avec une structure compatible avec votre projet.

**Structure attendue des données** :
- `data/raw/brain_mri/Training/...`
- `data/raw/brain_mri/Testing/...`


## 0) Pré-requis : rendre `src/` importable
Si vous voyez `ModuleNotFoundError: No module named 'pinkcc_ct_seg'`, c'est presque toujours parce que le notebook n'a pas `src/` dans le chemin d'import.

Deux solutions :
1) Lancer Jupyter via Poetry : `poetry run jupyter lab`
2) Utiliser le *fallback* ci-dessous (ajout de `src/` au `sys.path`).


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)


PROJECT_ROOT: C:\Users\Ange\OneDrive\Desktop\Brain_tumor_mri_classification\notebooks
SRC_DIR: C:\Users\Ange\OneDrive\Desktop\Brain_tumor_mri_classification\notebooks\src


## 1) Imports


In [2]:
import time
import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import matplotlib.pyplot as plt





In [3]:
import sys
from pathlib import Path

# Cherche la racine du projet (celle qui contient le dossier "src")
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    # si tu as lancé Jupyter depuis /notebooks, on remonte d'un niveau
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
print("PROJECT_ROOT =", PROJECT_ROOT)
print("SRC_DIR exists =", SRC_DIR.exists())

sys.path.insert(0, str(SRC_DIR))
print("src ajouté au sys.path ✅")
# Import depuis votre projet
from pinkcc_ct_seg.data.dataset import BrainMRIDataset

print("torch:", torch.__version__)

PROJECT_ROOT = C:\Users\Ange\OneDrive\Desktop\Brain_tumor_mri_classification
SRC_DIR exists = True
src ajouté au sys.path ✅
torch: 2.10.0+cpu


## 2) Configuration


In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

TRAIN_DIR = Path("data/raw/brain_mri/Training")
TEST_DIR  = Path("data/raw/brain_mri/Testing")

assert TRAIN_DIR.exists(), f"TRAIN_DIR introuvable: {TRAIN_DIR.resolve()}"
assert TEST_DIR.exists(),  f"TEST_DIR introuvable: {TEST_DIR.resolve()}"


device = cpu


## 3) Transforms


In [5]:
IMG_SIZE = 224

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])


## 4) Dataset + split Train/Val (stratifié)
Labels : 0 = `no_tumor`, 1 = autres classes


In [6]:
full_train_ds = BrainMRIDataset(TRAIN_DIR, transform=train_tfms)

labels = [full_train_ds.samples[i][1] for i in range(len(full_train_ds))]
dist = Counter(labels)
print("Nb images Training:", len(full_train_ds))
print("Distribution (Training):", dist)

idx_all = np.arange(len(full_train_ds))
train_idx, val_idx = train_test_split(
    idx_all,
    test_size=0.2,
    random_state=SEED,
    stratify=labels
)

train_ds = Subset(full_train_ds, train_idx)

# Val avec eval_tfms
full_train_eval_ds = BrainMRIDataset(TRAIN_DIR, transform=eval_tfms)
val_ds = Subset(full_train_eval_ds, val_idx)

print("Train size:", len(train_ds), "Val size:", len(val_ds))


Nb images Training: 2870
Distribution (Training): Counter({1: 2475, 0: 395})
Train size: 2296 Val size: 574


## 5) DataLoaders + gestion du déséquilibre (class weights)


In [7]:
BATCH_SIZE = 32
NUM_WORKERS = 2  # si souci sous Windows/WSL -> mettre 0

train_labels = [full_train_ds.samples[i][1] for i in train_idx]
c = Counter(train_labels)

w0 = 1.0 / c[0]
w1 = 1.0 / c[1]
class_weights = torch.tensor([w0, w1], dtype=torch.float32, device=device)

print("Train dist:", c)
print("class_weights:", class_weights.tolist())

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

test_ds = BrainMRIDataset(TEST_DIR, transform=eval_tfms)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

x, y = next(iter(train_loader))
print("Batch x:", x.shape, "Batch y:", y.shape, "dtype:", y.dtype, "min/max:", y.min().item(), y.max().item())


Train dist: Counter({1: 1980, 0: 316})
class_weights: [0.0031645570416003466, 0.0005050505278632045]
Batch x: torch.Size([32, 3, 224, 224]) Batch y: torch.Size([32]) dtype: torch.int64 min/max: 0 1


## 6) Modèle baseline (SmallCNN)


In [8]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 112

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 56

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 28
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (IMG_SIZE//8) * (IMG_SIZE//8), 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SmallCNN(num_classes=2).to(device)
model


SmallCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=100352, out_features=256, bias=

## 7) Loss / Optimizer / Scheduler


In [9]:
lr = 1e-3
epochs = 8

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)


## 8) Fonctions train/eval


In [10]:
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    all_preds, all_targets = [], []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        if is_train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * x.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(y.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, zero_division=0)

    return avg_loss, acc, f1


## 9) Entraînement + meilleur modèle (val F1)


In [ ]:
best_f1 = -1.0
best_state = None

history = {"train_loss": [], "train_acc": [], "train_f1": [],
           "val_loss": [], "val_acc": [], "val_f1": []}

for epoch in range(1, epochs + 1):
    t0 = time.time()

    train_loss, train_acc, train_f1 = run_one_epoch(model, train_loader, criterion, optimizer=optimizer)
    val_loss, val_acc, val_f1 = run_one_epoch(model, val_loader, criterion, optimizer=None)

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(f"Epoch {epoch:02d}/{epochs} | "
          f"train loss {train_loss:.4f} acc {train_acc:.3f} f1 {train_f1:.3f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.3f} f1 {val_f1:.3f} | "
          f"{(time.time()-t0):.1f}s")

print("Best val F1:", best_f1)


Epoch 01/8 | train loss 5.6096 acc 0.779 f1 0.859 | val loss 0.8230 acc 0.794 f1 0.867 | 514.8s


## 10) Courbes rapides


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.legend()
plt.title("Loss")
plt.show()

plt.figure(figsize=(8,4))
plt.plot(history["train_f1"], label="train_f1")
plt.plot(history["val_f1"], label="val_f1")
plt.legend()
plt.title("F1-score")
plt.show()


## 11) Évaluation Test + matrice de confusion


In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

test_loss, test_acc, test_f1 = run_one_epoch(model, test_loader, criterion, optimizer=None)
print(f"TEST | loss {test_loss:.4f} acc {test_acc:.3f} f1 {test_f1:.3f}")

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        y_pred.append(preds)
        y_true.append(y.numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

print("\nClassification report (0=no_tumor, 1=tumor):")
print(classification_report(y_true, y_pred, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix:\n", cm)

plt.figure(figsize=(4,4))
plt.imshow(cm)
plt.title("Confusion matrix")
plt.xlabel("Pred")
plt.ylabel("True")
plt.xticks([0,1], ["no_tumor", "tumor"])
plt.yticks([0,1], ["no_tumor", "tumor"])
for (i,j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")
plt.show()


## 12) Sauvegarde du modèle


In [ ]:
out_dir = Path("models")
out_dir.mkdir(parents=True, exist_ok=True)

ckpt = {
    "model_name": "SmallCNN",
    "img_size": IMG_SIZE,
    "state_dict": model.state_dict(),
    "best_val_f1": best_f1,
    "class_weights": class_weights.detach().cpu().tolist(),
    "seed": SEED,
}



ckpt_path = out_dir / "smallcnn_binary_baseline.pt"
torch.save(ckpt, ckpt_path)
print("Saved:", ckpt_path.resolve())
